In [ ]:
# =========================================================
# 0) Imports
# =========================================================
import polars as pl              # fast DataFrame engine
import pandas as pd # for scoring
import numpy as np
from pathlib import Path
import lightgbm as lgb
from   scipy.stats import rankdata   # for row-wise ranking
import pyarrow as pa

SOLUTION_NULL_FILLER = -999999
ROW_ID_COL           = "date_id"      # change if the column is named differently


######
# metrics from mitsui https://www.kaggle.com/code/metric/mitsui-co-commodity-prediction-metric/notebook
######
def rank_correlation_sharpe_ratio(merged_df: pd.DataFrame) -> float:
    """
    Calculates the rank correlation between predictions and target values,
    and returns its Sharpe ratio (mean / standard deviation).

    :param merged_df: DataFrame containing prediction columns (starting with 'prediction_')
                      and target columns (starting with 'target_')
    :return: Sharpe ratio of the rank correlation
    :raises ZeroDivisionError: If the standard deviation is zero
    """
    prediction_cols = [col for col in merged_df.columns if col.startswith('prediction_')]
    target_cols = [col for col in merged_df.columns if col.startswith('target_')]

    def _compute_rank_correlation(row):
        non_null_targets = [col for col in target_cols if not pd.isnull(row[col])]
        matching_predictions = [col for col in prediction_cols if col.replace('prediction', 'target') in non_null_targets]
        if not non_null_targets:
            raise ValueError('No non-null target values found')
        if row[non_null_targets].std(ddof=0) == 0 or row[matching_predictions].std(ddof=0) == 0:
            raise ZeroDivisionError('Denominator is zero, unable to compute rank correlation.')
        return np.corrcoef(row[matching_predictions].rank(method='average'), row[non_null_targets].rank(method='average'))[0, 1]

    daily_rank_corrs = merged_df.apply(_compute_rank_correlation, axis=1)
    std_dev = daily_rank_corrs.std(ddof=0)
    if std_dev == 0:
        raise ZeroDivisionError('Denominator is zero, unable to compute Sharpe ratio.')
    sharpe_ratio = daily_rank_corrs.mean() / std_dev
    return float(sharpe_ratio)


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculates the rank correlation between predictions and target values,
    and returns its Sharpe ratio (mean / standard deviation).
    """
    del solution[row_id_column_name]
    del submission[row_id_column_name]
    assert all(solution.columns == submission.columns)

    submission = submission.rename(columns={col: col.replace('target_', 'prediction_') for col in submission.columns})

    # Not all securities trade on all dates, but solution files cannot contain nulls.
    # The filler value allows us to handle trading halts, holidays, & delistings.
    solution = solution.replace(SOLUTION_NULL_FILLER, None)
    return rank_correlation_sharpe_ratio(pd.concat([solution, submission], axis='columns'))




# =========================================================
# 2) Read data with Polars
# =========================================================
#DATA_DIR = Path("/kaggle/input/mitsui-commodity-prediction-challenge")
# local path
DATA_DIR = Path("/home/ubuntu/repos/kaggle-mitsui/data")

# ---------- load ----------
feat_pl   = pl.read_csv(DATA_DIR / "train.csv")          # features   (has date_id)
label_pl  = pl.read_csv(DATA_DIR / "train_labels.csv")   # targets    (has date_id)
test_pl   = pl.read_csv(DATA_DIR / "test.csv")           # test feats (has date_id)

# ---------- join on `date_id` ----------
train_pl = feat_pl.join(label_pl, on="date_id", how="inner")

# ---------- column sets ----------
target_cols  = [c for c in train_pl.columns if c.startswith("target_")]
feature_cols = [c for c in train_pl.columns
                if c not in target_cols + ["date_id"]]


# fill all feature nulls with 0.0  (or median/mean per column)
train_pl = train_pl.with_columns(
    [pl.col(c).fill_null(0.0) for c in feature_cols]
)
test_pl  = test_pl.with_columns(
    [pl.col(c).fill_null(0.0) for c in feature_cols]
)

# ---------- numpy matrices for LightGBM ----------
X_train = train_pl.select(feature_cols).to_numpy()
y_dict  = {t: train_pl.select(t).to_numpy().ravel().astype(float) 
           for t in target_cols}

X_test  = test_pl.select(feature_cols).to_numpy()

# check the data
print("train_pl shape:", train_pl.shape)      # rows, cols
print("X_train shape:", X_train.shape, X_train.dtype)  # (n_rows, n_features)
print("Example target 0 shape:", y_dict[target_cols[0]].shape)



assert not np.isnan(X_train).any(),  "NaNs in features!"
assert X_train.dtype in (np.float32, np.float64), "LightGBM wants float"


# ---------- LightGBM params ----------
params = dict(
    objective        = "regression",
    boosting_type    = "gbdt",
    n_estimators     = 4000,
    learning_rate    = 0.02,
    num_leaves       = 127,
    feature_fraction = 0.8,
    subsample        = 0.7,
    lambda_l2        = 2e-3,
    random_state     = 42,
    verbose          = -1,
    n_jobs           = 16,            # 2 vCPU on Kaggle
)

# uncomment below to use GPU training
#params.update({
#    "device_type":     "gpu",
#    "max_bin":         255,
#    "gpu_platform_id": 0,
#    "gpu_device_id":   0,
#})

# ---------- train one model per target ----------
#  the built-in callback (LightGBM ≥4.0)
stop_cb = lgb.early_stopping(
    stopping_rounds=200,
    first_metric_only=False,   # or True if you monitor the first metric only
    verbose=False,
)

print("debugging")
models = {}
split = int(0.8 * X_train.shape[0])         # 80 % → train, 20 % → val
train_idx = slice(0, split)                 # rows 0 … split-1
val_idx   = slice(split, None)              # rows split … end


for k, tgt in enumerate(target_cols, 1):
    dtrain = lgb.Dataset(X_train[train_idx], label=y_dict[tgt][train_idx])
    dval   = lgb.Dataset(X_train[val_idx],   label=y_dict[tgt][val_idx])
    models[tgt] = lgb.train(
        params,
        dtrain,
        valid_sets=[dval],
        callbacks=[stop_cb],
    )
    if k % 25 == 0:          # progress print every 25 models
        print(f"{k}/{len(target_cols)}  best_iter={models[tgt].best_iteration:4}")


print("debugging")
# ---------- predict ----------
pred_dict = {"date_id": test_pl["date_id"]}
for tgt in target_cols:
    pred_col                       = tgt.replace("target_", "prediction_")
    pred_dict[pred_col]            = models[tgt].predict(X_test)

submission_pl = pl.DataFrame(pred_dict)
submission_pl.write_csv("lightgbm_polars_submission.csv")


# =========================================================
# 7) Optional: quick local validation using an 80/20 split
# =========================================================
from sklearn.model_selection import train_test_split

# 1. Create 80/20 hold-out split
idx = np.arange(len(train_pl))
tr_idx, v_idx = train_test_split(idx, test_size=0.2, shuffle=False)

# ------------------------------
# 2) Build validation preds
# ------------------------------
# Use the *same* target_* column names here:
val_pred_dict = {
    "date_id": train_pl["date_id"].to_numpy()[v_idx]
}
for tgt in target_cols:
    # NOTE: key is still 'target_X', not 'prediction_X'
    val_pred_dict[tgt] = models[tgt].predict(X_train[v_idx])

# 3. Wrap predictions in a Polars DataFrame
val_pred_pl = pl.DataFrame(val_pred_dict)

# 4. Extract validation solution rows using Polars slice
val_solution_pl = train_pl.select(["date_id"] + target_cols).slice(v_idx[0], len(v_idx))

# 5. Convert both solution and predictions to Pandas for scoring
val_score = score(
    solution=val_solution_pl.to_pandas(),
    submission=val_pred_pl.to_pandas(),
    row_id_column_name="date_id"
)

# 6. Report result
print(f"Local Sharpe score (hold-out): {val_score:.4f}")
